# Finetuning MDSEM (Stage 1 of UMSS)

In [ ]:
import numpy as np
import keras
import matplotlib.pyplot as plt
import sys,os
from keras.models import Model
import tensorflow as tf
from keras.utils import Sequence
from keras.optimizers import Adam
import cv2
from keras.callbacks import ModelCheckpoint, LearningRateScheduler
from PIL import Image
from IPython.display import clear_output
from copy import deepcopy
import tqdm 
import math

sys.path.append('../src')

from data_loading import load_datasets_multiduration
from util import get_model_by_name, create_losses

#from eval import *
from sal_imp_utilities import padding, preprocess_images, reverse_preprocess
from cb import InteractivePlot

%load_ext autoreload
%autoreload 2

# Check GPU status

In [ ]:
%%bash
nvidia-smi

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = '0'
os.environ["CUDA_VISIBLE_DEVICES"]

# Load data

In [ ]:
# FILL THESE IN
#dataset = "salicon_md"
#dataset = "codecharts"
#dataset = "massvis_md"
#dataset = "massvis_bucket_500_2000_5000"
dataset = "massvis_bucket_500_3000_5000"
#dataset = "massvis_bucket_global_500_2000_5000"
bp = "/netpool/work/gpu-3/users/wangyo/datasets"

In [ ]:
times = [500, 3000, 5000]
#times = [3000, 5000, 10000]

data = load_datasets_multiduration(dataset, times, bp=bp)

# Model and training params

In [ ]:
# FILL THESE IN: set training parameters 
ckpt_savedir = "../ckpt"

# train
load_weights = True
# md-sem trained on salicon-md. Fine tune this on new code charts w/ccm. 
# weightspath = "/netpool/work/gpu-2/users/wangyo/projects/mdsem/ckpt/salicon_md_kl10cc-5nss-10ccmatch3_ep09_valloss-11.6482.hdf5"
# weightspath = "/mnt/localssd2/predimportance/predimportance_shared/models/ckpt/xception_se_lstm/xception_se_lstm_salicon_md_fixations_3match10kl-5cc-1nss_ep04_valloss-2.2668.hdf5"
# weightspath = "/data/vision/oliva/scratch/anelise/predimportance/models/ckpt/sam_resnet_salmd/salicon_md_fixations_6bc4kl-3cc-10nss_ep01_valloss-20.3203.hdf5"
# weightspath = "/netpool/work/gpu-2/users/wangyo/projects/mdsem/model/ckpt_mdsem/salicon_md_kl10cc-5nss-10ccmatch3_ep10_valloss-17.1473.hdf5"
weightspath = "../model/mdsem_codecharts0_cameraready_weights.hdf5"

batch_size = 8
init_lr = 0.0001
lr_reduce_by = .1
reduce_at_epoch = 3
n_epochs = 9

opt = Adam(lr=init_lr)

# losses is a dictionary mapping loss names to weights 
#losses = {
     #'kl': 10,
     #'cc': -5,
     #'nss': -10,
     #'ccmatch': 3
#}

losses = {
     'kl': 10,
     'cc': -5,
     'nss': -1,
     'ccmatch': 3
}


model_name = "md_sem"
model_inp_size = (240, 320)
model_out_size = (480, 640)
n_timesteps = len(times)

In [ ]:
# get model 
model_params = {
    'input_shape': model_inp_size + (3,),
    'n_outs': len(losses),
    'nb_timestep': n_timesteps
}
model_func, mode = get_model_by_name(model_name)
model = model_func(**model_params)

if load_weights: 
    model.load_weights(weightspath)
    print("Loaded")

In [ ]:
# set up data generation and checkpoints
if not os.path.exists(ckpt_savedir): 
    os.makedirs(ckpt_savedir)
    
# sort the losses so that those that use a fixmap are last, by convention
l, lw, l_str, n_heatmaps = create_losses(losses, model_out_size)
n_fixmaps = len(l) - n_heatmaps
print("Loss string", l_str)

# Generators
gen_train = MultidurationGenerator(
                img_filenames=data['img_files_train'], 
                map_filenames=data['map_files_train'], 
                fix_filenames=data['fix_files_train'], 
                batch_size=batch_size, 
                mode=mode,
                read_npy=False, # True for salicon-md, False for codecharts
                img_size=model_inp_size, 
                map_size=model_out_size,
                shuffle=True, 
                augment=False, 
                n_output_maps=n_heatmaps,
                n_output_fixs=n_fixmaps,
                fix_as_mat=data.get('fix_as_mat', False),
                fix_key=data.get('fix_key', ''))

gen_val = MultidurationGenerator(
            img_filenames=data['img_files_val'], 
            map_filenames=data['map_files_val'], 
            fix_filenames=data['fix_files_val'], 
            batch_size=1, 
            mode=mode,
            read_npy=False, # True for salicon-md, False for codecharts
            img_size=model_inp_size, 
            map_size=model_out_size,
            shuffle=False, 
            augment=False, 
            n_output_maps=n_heatmaps,
            n_output_fixs=n_fixmaps,
            fix_as_mat=data.get('fix_as_mat', False),
            fix_key=data.get('fix_key', '')
        )

# Callbacks

# where to save checkpoints
filepath = os.path.join(ckpt_savedir, dataset + "_" + l_str + '_ep{epoch:02d}_valloss{val_loss:.4f}.hdf5')
print("Checkpoints will be saved with format %s" % filepath)

cb_chk = ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_weights_only=True, period=1)
cb_plot = InteractivePlot()

def step_decay(epoch):
    lrate = init_lr * math.pow(lr_reduce_by, math.floor((1+epoch)/reduce_at_epoch))
    if epoch%reduce_at_epoch:
        print('Reducing lr. New lr is:', lrate)
    return lrate
cb_sched = LearningRateScheduler(step_decay)

cbs = [cb_chk, cb_sched, cb_plot]

In [ ]:
#test the generatpng
img, outs = gen_train.__getitem__(1)
print("batch size: %d. Num inputs: %d. Num outputs: %d." % (batch_size, len(img), len(outs)))
print(outs[0].shape)

# Train

In [ ]:
model.compile(optimizer=opt, loss=l, loss_weights=lw)

print('Ready to train')
model.fit_generator(gen_train, epochs=n_epochs, verbose=1, callbacks=cbs, validation_data=gen_val, max_queue_size=10,  workers=5)

# Evaluate

In [ ]:
image_path = "/path/to/test"
list_img_targets = natsorted(glob.glob(image_path + '/*.*g'))
images = preprocess_images(list_img_targets, model_inp_size[0], model_inp_size[1])

In [ ]:
preds_mdsem = model.predict(images)
print(np.array(preds_mdsem).shape)

In [ ]:
times = [500, 2000, 5000] 
ntimes = len(times)

def transparent_cmap(cmap, N=255):
    "Copy colormap and set alpha values"
    mycmap = cmap
    mycmap._init()
    mycmap._lut[:,-1] = np.linspace(0.5, 0.8, N+4)
    #mycmap._lut[:,-1] = np.linspace(0, 0.8, N+4) # full transparent
    return mycmap


vis_path = '/path/to/vis/'
out_path = '/path/to/pred/'

pred_batch=[]
mycmap = transparent_cmap(plt.cm.jet)
for i in range(len(images)):
    
    # Open original image to get width and height
    im = Image.open(list_img_targets[i])
    width, height = im.size


    for t in range(ntimes):
        plt.imshow(im)
        a = preds_mdsem[0][i,t,:,:,0]

        if width*shape_r > height*shape_c:
            crop_pred = preds_mdsem[0][i,t,:,:,0][int(shape_r - height/width*shape_c):int(shape_r + height/width*shape_c),:]
        else:
            crop_pred = preds_mdsem[0][i,t,:,:,0][:,int(shape_c - width/height*shape_r):int(shape_c + width/height*shape_r)]

        pred = Image.fromarray(np.uint8(crop_pred * 255)).resize(im.size)
        plt.imshow(pred, alpha=0.5, cmap=mycmap)
        plt.axis('off')

        file_name = list_img_targets[i].split('/')[-1].replace('.',f"_{times[t]}.")
        pred.save(out_path + file_name)
        plt.savefig(vis_path + file_name, bbox_inches='tight', pad_inches=0, dpi=300)
        plt.close()
